In [15]:
from firecrawl import AsyncFirecrawlApp

import os
from dotenv import load_dotenv

In [16]:
dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path)

API_KEY = os.getenv("GOOGLE_API_KEY")
FIRECRAWL_KEY = os.getenv("FIRECRAWL_API_KEY")

In [17]:
url = 'https://fybelle.com'

In [18]:
async def main():
    app = AsyncFirecrawlApp(api_key=FIRECRAWL_KEY)
    response = await app.map_url(
        url=url,
        include_subdomains=True
    )
    return response

# asyncio.run(main())

In [19]:
links = await main()
links = links.links
links

['https://fybelle.com',
 'https://fybelle.com/blog',
 'https://fybelle.com/brazilian-laser-hair-removal-what-is-the-safest-most-affordable-solution',
 'https://fybelle.com/best-at-home-laser-hair-removal',
 'https://fybelle.com/how-much-does-laser-hair-removal-cost-2025-guide',
 'https://fybelle.com/facial-hair-removal-5-safe-effective-methods',
 'https://fybelle.com/what-is-the-best-ipl-hair-removal',
 'https://fybelle.com/do-led-face-masks-work-the-science-benefits-and-best-picks',
 'https://fybelle.com/does-ipl-hair-removal-really-work',
 'https://fybelle.com/shop',
 'https://fybelle.com/product/led-light-therapy-face-mask',
 'https://fybelle.com/product/fybelle-ice-ipl-hair-removal-handset',
 'https://fybelle.com/category/ipl',
 'https://fybelle.com/category/led',
 'https://fybelle.com/my-account',
 'https://fybelle.com/contraindictions',
 'https://fybelle.com/checkout',
 'https://fybelle.com/about-us',
 'https://fybelle.com/intellectual-property-rights',
 'https://fybelle.com/priv

In [20]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import load_tools, initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

import re
import json


SERP_API_KEY = os.getenv('SERP_API_KEY')


llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=API_KEY, temperature=0)
memory = ConversationBufferMemory(memory_key='chat_history')
tools = load_tools(['serpapi','llm-math'], llm=llm, serpapi_api_key=SERP_API_KEY)
agent = initialize_agent(tools=tools,llm=llm, agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION, memory=memory,verbose=True, handle_parsing_errors=True)



filter_message = f"""
Eres un especialsita en recopilacion de datos de productos de tiendas online. Tu objetivo es a partir de una lista de urls definin cuales de los links son de productos de la tienda

Tu objetivo sera descartar todas las url que no sean de productos. Miraras en la lista de las urls y escojeras SOLO LAS QUE SEAN DE PRODUCTOS. Ten en cuenta que los productos pueden estar en varios idiomas

Es posible que exsistan productos repetidos en diferentes links o idiomas, deberas dercartar los productos que esten repetidos y quedarte solo con uno de ellos, si estan en diferentes idiomas escoje solo los que esten en español

Lista de URLs : 
{links}

La respuesta se usara para meter los datos en una base de datos asi que debe ser ÚNICAMENTE el siguiente objeto JSON, sin ningún texto antes o después, ni explicaciones, ni comentarios.

{{
"links" : [....]
}}

"""


data = agent.invoke(filter_message)['output']

json_match = re.search(r'\{.*\}', data, re.DOTALL)
if json_match:
    data = json_match.group(0)

data = json.loads(data)
product_links = data['links']



> Entering new AgentExecutor chain...
Could not parse LLM output: ````json
{
"links": [
"https://fybelle.com/product/led-light-therapy-face-mask",
"https://fybelle.com/product/fybelle-ice-ipl-hair-removal-handset"
]
}
````
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Observation: Invalid or incomplete response
Thought:Do I need to use a tool? No
AI: ```json
{
"links": [
"https://fybelle.com/product/led-light-therapy-face-mask",
"https://fybelle.com/product/fybelle-ice-ipl-hair-removal-handset"
]
}
```

> Finished chain.


In [21]:
len(product_links)

2

In [22]:
app = AsyncFirecrawlApp(api_key=FIRECRAWL_KEY)
data = await app.batch_scrape_urls(
    product_links, 
    formats=['json'],
    json_options={
        'prompt' : 'Extract the title, sourceURL and Main Image from the page',
        'schema': {
            'type': 'json',
            'properties': {
                'title': {'type': 'string'},
                'sourceURL': {'type': 'string'},
                'imageURL' : {'type': 'string'}
            },
            'required': ['title', 'sourceURL', 'imageURL']
        }
    }
    )                                                                                                                                                                                              

In [23]:
scraped_webs = data.data
scraped_webs

[FirecrawlDocument(url=None, markdown=None, html=None, rawHtml=None, links=None, extract=None, json=None, screenshot=None, metadata={'og:title': 'Fybelle ICE IPL Hair Removal Handset - Fybelle', 'robots': 'index, follow, max-snippet:-1, max-video-preview:-1, max-image-preview:large', 'generator': ['WordPress 6.8.1', 'Elementor 3.21.4; features: e_optimized_assets_loading, e_optimized_css_loading, e_font_icon_svg, additional_custom_breakpoints; settings: css_print_method-external, google_font-enabled, font_display-swap'], 'twitter:data2': 'In stock', 'ogSiteName': 'Fybelle', 'og:image:secure_url': 'https://fybelle.com/wp-content/uploads/2023/03/Pic1.jpg', 'viewport': 'width=device-width, initial-scale=1', 'og:image': 'https://fybelle.com/wp-content/uploads/2023/03/Pic1.jpg', 'og:image:width': '800', 'og:locale': 'en_US', 'description': 'Fybelle IPL Hair Removal handset that will change the way you do hair removal forever PAIN-FREE - The newly added Ice Cooling Technology acts as an ice 

In [24]:
products_message = f"""
Eres un especialsita en recopilacion de datos de productos de tiendas online. Tu objetivo es a partir de una lista de paginas de productos de un sitio web en markdown sacar toda la informacion necesaria de cada uno de los productos de la lista. Estan como FireCralw Document

Tu primer objetivo es descartar de las lista todas las paginas obsoletas o que vacias

Tu segundo objetivo es sacar la informacion de cada uno de los productos para esto tendras que leer los datos de todas las paginas de productos que has flitrado previamente

Que informacion sacaras?:
1. Nombre del Producto
2. Url del producto (es el sourceURL de la pagina)
3. Link a la imagen del producto (estara en algun "og:image:, habra varios links, asegurate que sea el link correcto)
4. Industria en la que se desarrolla el producto (Cosmetica, Salud, Telefonia, Suplementos, etc...)

Lista de paginas : 
{scraped_webs}

La respuesta se usara para meter los datos en una base de datos asi que debe ser ÚNICAMENTE el siguiente objeto JSON, sin ningún texto antes o después, ni explicaciones, ni comentarios:

{{
"producto-1": {{
        "nombre-producto": "...",
        "url-producto": "...",
        "url-imagen" : "...",
        "industria" : "..."
        }}
"producto-2": {{
        "nombre-producto": "...",
        "url-producto": "...",
        "url-imagen" : "...",
        "industria" : "..."
        }}
...
}}


El JSON devuelvelo sin saltos de linea, lo necescito asi

"""


data = agent.invoke(products_message)['output']

json_match = re.search(r'\{.*\}', data, re.DOTALL)
if json_match:
    data = json_match.group(0)

data = json.loads(data)




> Entering new AgentExecutor chain...
Could not parse LLM output: ````json
{
"producto-1": {
        "nombre-producto": "Fybelle ICE IPL Hair Removal Handset",
        "url-producto": "https://fybelle.com/product/fybelle-ice-ipl-hair-removal-handset",
        "url-imagen" : "https://fybelle.com/wp-content/uploads/2023/03/Pic1.jpg",
        "industria" : "Cosmetica"
        },
"producto-2": {
        "nombre-producto": "LED Light Therapy Face Mask",
        "url-producto": "https://fybelle.com/product/led-light-therapy-face-mask",
        "url-imagen" : "https://fybelle.com/wp-content/uploads/2025/02/fybelle-feeds-ads-30.png",
        "industria" : "Cosmetica"
        }
}
````
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Observation: Invalid or incomplete response
Thought:Do I need to use a tool? No
AI: ```json{"producto-1":{"nombre-producto":"Fybelle ICE IPL Hair Removal Handset","url-producto":"https://fybelle.com/produ

In [26]:
data

{'producto-1': {'nombre-producto': 'Fybelle ICE IPL Hair Removal Handset',
  'url-producto': 'https://fybelle.com/product/fybelle-ice-ipl-hair-removal-handset',
  'url-imagen': 'https://fybelle.com/wp-content/uploads/2023/03/Pic1.jpg',
  'industria': 'Cosmetica'},
 'producto-2': {'nombre-producto': 'LED Light Therapy Face Mask',
  'url-producto': 'https://fybelle.com/product/led-light-therapy-face-mask',
  'url-imagen': 'https://fybelle.com/wp-content/uploads/2025/02/fybelle-feeds-ads-30.png',
  'industria': 'Cosmetica'}}